# UR10 Simple Reach — RTDE Control Loop

Deploy the trained 6-DOF reaching policy on a real UR10 (or URSim) via RTDE.

**Observation (18D):** `q[6], qd[6], tcp_pos[3], target_pos[3]`  
**Action (6D):** delta joint positions, scaled by `action_scale=0.04`  
**Control:** `ctrl_next = clip(ctrl_prev + action_scale * action, lowers, uppers)`

## 1 — Backend setup

In [15]:
import os
import platform
import subprocess

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"

system = platform.system()
if system == "Darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
    print("macOS — MUJOCO_GL=glfw, JAX on CPU")
elif subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
    os.environ["MUJOCO_GL"] = "egl"
    print("NVIDIA GPU detected — MUJOCO_GL=egl")
else:
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
    print("No GPU — MUJOCO_GL=osmesa, JAX on CPU")

macOS — MUJOCO_GL=glfw, JAX on CPU


## 2 — Imports

In [16]:
import json
import time

import jax
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np

from URSim_RTDE_dependencies import URSimRTDESimpleReach

print("JAX devices:", jax.devices())

JAX devices: [CpuDevice(id=0)]


## 3 — Configuration

In [17]:
HOST = "127.0.0.1"          # URSim or real robot IP

# Paths
MODEL_PATH = "../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_reach.xml"
POLICY_PATH = "../../evaluation/downloaded_policies/simple_reach_policy"

# Target position in world frame (meters) — must be within training workspace
# Training range: x=[0.3,0.8], y=[-0.4,0.4], z=[0.2,0.8]
TARGET_POS = np.array([0.5, 0.0, 0.5])

# Control loop settings (must match training: ctrl_dt=0.02 -> 50 Hz)
CONTROL_HZ = 50.0
ACTION_SCALE = 0.04
TIMEOUT_S = 10.0

# Start pose — "low_home" keyframe from mjx_reach.xml
Q_START = [0, -1.7, 2.25, -2.15, -1.5, -1.5]

# servoj parameters
LOOKAHEAD_TIME = 0.1
GAIN = 300

## 4 — Connect to robot & verify

In [18]:
robot = URSimRTDESimpleReach(host=HOST)
robot.connect()
robot.print_feedback()

q      = [0.9584, 2.3035, -0.1571, -0.5782, -2.9239, -3.3644]
qd     = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.3729, 0.4682, -0.4612]


## 5 — Load policy & MuJoCo FK model

In [19]:
robot.load_policy_fn(policy_path=POLICY_PATH)
robot.init_fk_model(MODEL_PATH)

Loaded policy: obs=18, act=6


AttributeError: 'URSimRTDESimpleReach' object has no attribute 'init_fk_model'

## 6 — Verify policy in simulation (same as UR10_SimpleReach.ipynb chunk 10)

In [ ]:
import mujoco
from mujoco_playground import registry

loaded_env = registry.load(robot._meta["env_name"])
jit_reset = jax.jit(loaded_env.reset)
jit_step  = jax.jit(loaded_env.step)

mj_model = mujoco.MjModel.from_xml_path(MODEL_PATH)
mj_data  = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model, height=480, width=640)

cam = mujoco.MjvCamera()
cam.type      = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = [0.4, 0.0, 0.4]
cam.distance  = 1.8
cam.azimuth   = 130
cam.elevation = -20

state = jit_reset(jax.random.PRNGKey(42))

frames_sim = []
rewards_sim = []
for t in range(150):
    action = robot._policy_fn(state.obs)
    state = jit_step(state, action)
    rewards_sim.append(float(state.reward))

    mj_data.qpos[:]      = np.array(state.data.qpos)
    mj_data.qvel[:]      = np.array(state.data.qvel)
    mj_data.ctrl[:]      = np.array(state.data.ctrl)
    mj_data.mocap_pos[:] = np.array(state.data.mocap_pos)
    mujoco.mj_forward(mj_model, mj_data)

    if t % 2 == 0:
        renderer.update_scene(mj_data, camera=cam)
        frames_sim.append(renderer.render().copy())

    if float(state.done) > 0.0:
        print(f"Episode ended at step {t+1}")
        break

target_sim = np.array(state.obs[15:18])
tcp_sim = np.array(state.obs[12:15])
print(f"{len(frames_sim)} frames | Final dist: {np.linalg.norm(target_sim - tcp_sim)*100:.1f} cm | Reward: {sum(rewards_sim):.1f}")
media.show_video(frames_sim, fps=30, height=480)

## 7 — JIT warmup

In [ ]:
dummy_obs = np.zeros((1, 18), dtype=np.float32)
t0 = time.perf_counter()
_ = robot._policy_fn(dummy_obs)
_.block_until_ready()
print(f"JIT warmup done in {time.perf_counter() - t0:.2f}s")

## 8 — Move to start pose

In [ ]:
robot.move_to_start(Q_START, a=1.5, v=1.0, timeout_s=15.0, tol=0.01)

## 9 — Run policy control loop

In [ ]:
df, stats = robot.run_policy_loop(
    target_pos=TARGET_POS,
    control_hz=CONTROL_HZ,
    timeout_s=TIMEOUT_S,
    action_scale=ACTION_SCALE,
    lookahead_time=LOOKAHEAD_TIME,
    gain=GAIN,
)
print(f"\nLogged {len(df)} steps")

## 10 — Timing statistics

In [ ]:
print("=== Summary ===")
robot.print_stats(stats)

print(f"\n=== Timing breakdown (mean / max) ===")
print(f"  RTDE receive : {df['obs_time_s'].mean()*1000:6.2f} / {df['obs_time_s'].max()*1000:6.2f} ms")
print(f"  Policy infer : {df['policy_time_s'].mean()*1000:6.2f} / {df['policy_time_s'].max()*1000:6.2f} ms")
print(f"  servoj send  : {df['send_time_s'].mean()*1000:6.2f} / {df['send_time_s'].max()*1000:6.2f} ms")
total_compute = df['obs_time_s'] + df['policy_time_s'] + df['send_time_s']
print(f"  Total compute: {total_compute.mean()*1000:6.2f} / {total_compute.max()*1000:6.2f} ms")
print(f"  Available dt : {1000/CONTROL_HZ:6.2f} ms")
print(f"  Overruns     : {stats.get('num_overruns', 0)}")

## 11 — Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df["time"], df["tcp_to_target_dist"] * 100)
axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("distance (cm)")
axes[0].set_title("TCP -> target distance"); axes[0].grid(True, alpha=0.3)

axes[1].plot(df["time"], df["loop_hz_true"], alpha=0.7)
axes[1].axhline(CONTROL_HZ, color="r", ls="--", label=f"target={CONTROL_HZ} Hz")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("Hz")
axes[1].set_title("Loop frequency"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(df["time"], df["obs_time_s"] * 1000, label="RTDE recv", alpha=0.7)
axes[2].plot(df["time"], df["policy_time_s"] * 1000, label="policy", alpha=0.7)
axes[2].plot(df["time"], df["send_time_s"] * 1000, label="servoj send", alpha=0.7)
axes[2].set_xlabel("time (s)"); axes[2].set_ylabel("ms")
axes[2].set_title("Timing per step"); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12 — MuJoCo video replay

In [ ]:
mj = robot.mujoco_init_model(
    xml_path=MODEL_PATH,
    height=480, width=640,
    cam_lookat=(0.4, 0.0, 0.4),
    cam_distance=1.8, cam_azimuth=130, cam_elevation=-20,
)

VIDEO_FPS = 30.0
frames, actual_fps = robot.render_video_from_log(mj, df, video_fps=VIDEO_FPS)
media.show_video(frames, fps=actual_fps, height=480)

## 13 — Frequency analysis

In [ ]:
print("=== Frequency analysis ===")
compute_ms = (df["obs_time_s"] + df["policy_time_s"] + df["send_time_s"]) * 1000
print(f"Mean compute time: {compute_ms.mean():.2f} ms")
print(f"Max  compute time: {compute_ms.max():.2f} ms")
print(f"P95  compute time: {compute_ms.quantile(0.95):.2f} ms")
print()
max_hz_mean = 1000.0 / compute_ms.mean()
max_hz_p95 = 1000.0 / compute_ms.quantile(0.95)
print(f"Max sustainable Hz (mean):  {max_hz_mean:.0f} Hz")
print(f"Max sustainable Hz (p95):   {max_hz_p95:.0f} Hz")
print(f"Training ctrl_dt = 0.02s -> 50 Hz")
print(f"Achieved loop Hz: {stats['mean_loop_hz_true']:.1f} Hz")

## 14 — Disconnect

In [ ]:
robot.disconnect()
print("Disconnected.")